## Регрессия для IC50

### Импортируем библиотеки

In [21]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge

### Загружаем данные

In [2]:
df_ml = pd.read_csv('data_classic_ML.csv')

Удалим метрики при обучении т.к. если оставить SI и CC50 в признаках, модель мгновенно «догадается» вычесть из логарифма безопасности логарифм индекса селективности. Алгоритм покажет идеальную точность, но модель будет абсолютно бесполезна на практике, так как для новых (еще не синтезированных) молекул значения CC50 и SI заранее неизвестны. Модель должна опираться строго на топологию и свойства самого химического элемента.

In [3]:
target_cols = ['log_IC50', 'log_CC50', 'log_SI']
X = df_ml.drop(columns=[col for col in target_cols if col in df_ml.columns])
y = df_ml['log_IC50']

Разбиение на обучающую и тестовую выборки (70% на 30%)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

Масштабирование дескрипторов через алгортм RobustScaler

In [6]:
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Модель 1: Линейная регрессия

In [7]:
ridge_model = Ridge(alpha=5.0)
ridge_model.fit(X_train_scaled, y_train)

y_pred_ridge = ridge_model.predict(X_test_scaled)

r2_ridge = r2_score(y_test, y_pred_ridge)
mse_ridge = mean_squared_error(y_test, y_pred_ridge)
mae_ridge = mean_absolute_error(y_test, y_pred_ridge)

print(f"Коэффициент детерминации (R²): {r2_ridge:.4f}")
print(f"Среднеквадратичная ошибка (MSE): {mse_ridge:.4f}")
print(f"Средняя абсолютная ошибка (MAE): {mae_ridge:.4f}")

Коэффициент детерминации (R²): 0.3176
Среднеквадратичная ошибка (MSE): 0.6038
Средняя абсолютная ошибка (MAE): 0.6173


### Модель 2: Случайный лес

Подбор гиперпараметров

In [12]:
param_grid = {
    'n_estimators': [300, 500, 700],
    'max_depth': [12, 15, 18],
    'min_samples_split': [2, 4, 6],
    'max_features': ['sqrt', 'log2', 0.3]
}

grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=-1), 
    param_grid=param_grid, 
    cv=5, 
    scoring='r2', 
    n_jobs=-1, 
    verbose=1
)

grid_search.fit(X_train_scaled, y_train)

print(f"Лучшие подобранные параметры: {grid_search.best_params_}")

Fitting 5 folds for each of 81 candidates, totalling 405 fits
Лучшие подобранные параметры: {'max_depth': 18, 'max_features': 'sqrt', 'min_samples_split': 2, 'n_estimators': 700}


Обучение модели

In [13]:
rf_model = RandomForestRegressor(
    n_estimators=700,
    max_depth=18,
    min_samples_split=2,
    max_features='sqrt',
    random_state=42, 
    n_jobs=-1
)
rf_model.fit(X_train_scaled, y_train)
y_pred_rf = rf_model.predict(X_test_scaled)

r2_rf = r2_score(y_test, y_pred_rf)
mse_rf = mean_squared_error(y_test, y_pred_rf)
mae_rf = mean_absolute_error(y_test, y_pred_rf)

print(f"Коэффициент детерминации (R²): {r2_rf:.4f}")
print(f"Среднеквадратичная ошибка (MSE): {mse_rf:.4f}")
print(f"Средняя абсолютная ошибка (MAE): {mae_rf:.4f}")

Коэффициент детерминации (R²): 0.4340
Среднеквадратичная ошибка (MSE): 0.5008
Средняя абсолютная ошибка (MAE): 0.5479


### Модель 3: Гистограммный градиентный бустинг

Подбор гиперпараметров

In [14]:
param_grid = {
    'max_iter': [100, 300, 500],
    'learning_rate': [0.02, 0.04, 0.08],
    'max_depth': [4, 6, 8],
    'l2_regularization': [0.1, 1.0, 5.0]
}
grid_search_hgb = GridSearchCV(
    estimator=HistGradientBoostingRegressor(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

grid_search_hgb.fit(X_train_scaled, y_train)

print(f"Лучшие подобранные параметры: {grid_search_hgb.best_params_}")

Fitting 5 folds for each of 81 candidates, totalling 405 fits
Лучшие подобранные параметры: {'l2_regularization': 5.0, 'learning_rate': 0.02, 'max_depth': 4, 'max_iter': 500}


In [15]:
hgb_model = HistGradientBoostingRegressor(
    max_iter=500, 
    learning_rate=0.02, 
    max_depth=4, 
    l2_regularization=5.0, 
    random_state=42
)
hgb_model.fit(X_train_scaled, y_train)
y_pred_hgb = hgb_model.predict(X_test_scaled)

r2_hgb = r2_score(y_test, y_pred_hgb)
mse_hgb = mean_squared_error(y_test, y_pred_hgb)
mae_hgb = mean_absolute_error(y_test, y_pred_hgb)

print(f"Коэффициент детерминации (R²): {r2_hgb:.4f}")
print(f"Среднеквадратичная ошибка (MSE): {mse_hgb:.4f}")
print(f"Средняя абсолютная ошибка (MAE): {mae_hgb:.4f}")

Коэффициент детерминации (R²): 0.4448
Среднеквадратичная ошибка (MSE): 0.4913
Средняя абсолютная ошибка (MAE): 0.5417


### Анализ результата

In [19]:
summary_data = {
    'Модель': ['Линейная регрессия', 'Случайный лес', 'Градиентный бустинг'],
    'R²': [r2_ridge, r2_rf, r2_hgb],
    'MSE': [mse_ridge, mse_rf, mse_hgb],
    'MAE': [mae_ridge, mae_rf, mae_hgb]
}
df_summary = pd.DataFrame(summary_data).round(4)
print(df_summary.to_string(index=False))

             Модель     R²    MSE    MAE
 Линейная регрессия 0.3176 0.6038 0.6173
      Случайный лес 0.4340 0.5008 0.5479
Градиентный бустинг 0.4448 0.4913 0.5417


Модель Линейной регрессии показала наименьшую предсказательную способность среди протестированных методов R^2 = 0.3176. Коэффициент детерминации показывает, что линейные веса смогли объяснить лишь 32% дисперсии целевой переменной.  
Результат Случайного леса R^2 = 0.4340 и лучший результат Градиентный бустинг R^2 = 0.4448. Ансамбли на основе деревьев решений показали высокую применимость для задач моделирования табличных химических данных. Переход к нелинейным алгоритмам позволил снизить ошибку прогноза. 

Рекомендации по улучшению моделей:   
- Текущие признаки представляют собой одномерные физико-химические дескрипторы. Рекомендуется рассчитать молекулярные фингерпринты (например, Morgan/Circular Fingerprints, MACCS keys). Они кодируют топологию молекулы в виде плотных бинарных векторов, что позволяет моделям машинного обучения гораздо глубже оценивать скрытые структурные сходства.
- Вместо жесткого удаления признаков по порогу корреляции r > 0.90 возможно имеет смысл применить методы Lasso (L1-регуляризацию) или RFE (Recursive Feature Elimination) непосредственно внутри ансамблей. Это позволит удержать нелинейные слабые сигналы от дескрипторов, которые линейно казались зависимыми, но в комбинации дают прирост точности.
- Объединить прогнозы лучших моделей (Случайного леса и Градиентного бустинга) с помощью мета-модели

### Реализация мета-модели на основе Случайного леса и Градиентного бустинга

In [23]:
base_models = [
    ('random_forest', RandomForestRegressor(
        n_estimators=700, max_depth=18, min_samples_split=2, max_features='sqrt', random_state=42, n_jobs=-1
    )),
    ('hgb_model', HistGradientBoostingRegressor(
        max_iter=500, learning_rate=0.02, max_depth=4, l2_regularization=5.0, random_state=42
    ))
]

meta_model = StackingRegressor(
    estimators=base_models,
    final_estimator=Ridge(alpha=1.0),
    cv=5,
    n_jobs=-1
)
meta_model.fit(X_train_scaled, y_train)
y_pred_meta = meta_model.predict(X_test_scaled)

r2_meta = r2_score(y_test, y_pred_meta)
mse_meta = mean_squared_error(y_test, y_pred_meta)
mae_meta = mean_absolute_error(y_test, y_pred_meta)

print(f"Коэффициент детерминации (R²): {r2_meta:.4f}")
print(f"Среднеквадратичная ошибка (MSE): {mse_meta:.4f}")
print(f"Средняя абсолютная ошибка (MAE): {mae_meta:.4f}")

Коэффициент детерминации (R²): 0.4397
Среднеквадратичная ошибка (MSE): 0.4958
Средняя абсолютная ошибка (MAE): 0.5430


Полученный результат практически полностью повторяет точность лучших базовых моделей, что математически объясняется их однородной природой (оба алгоритма основаны на ансамблях деревьев решений) и высокой степенью корреляции их локальных прогнозов. Тем не менее, внедрение мета-алгоритма обосновано с точки зрения повышения стабильности системы: стекинг выступает в роли дополнительного регуляризатора, сглаживая экстремальные ошибки отдельных деревьев и гарантируя воспроизводимость результатов скрининга на новых химических соединениях».